# Fast masked-node DAS (ones-carry -> tens digit)

Speed changes vs. the previous notebook:
1. **Orthonormal basis computed once per step** with batched QR (the old `orthogonal` parametrization was re-evaluated on every access, i.e. every micro-batch).
2. **Basis / mask gradients accumulated on detached leaves**, then pushed back through QR once.
3. **Vectorized hooks**: one gather/einsum per layer instead of a Python loop per node.
4. **Minibatch steps** (e.g. 64 pairs per step) instead of full-dataset epochs.
5. **fp16 autocast** (T4 tensor cores) for the transformer; intervention math stays fp32.
6. Source activations kept on GPU. Timing is printed after the first steps.

Keep `subspace_dim` small (1-8). Dim 128 per node is almost never what DAS wants and makes everything slow.

In [ ]:
!pip install -q transformer_lens

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 2.5 MB/s eta 0:00:00


In [ ]:
import time, math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformer_lens.model_bridge import TransformerBridge

model = TransformerBridge.boot_transformers("Qwen/Qwen3-1.7B", device="cuda")
model.eval()
model.requires_grad_(False)
device = next(model.parameters()).device
print("device:", device, "| layers:", model.cfg.n_layers, "| d_model:", model.cfg.d_model)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/25.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

device: cuda:0 | layers: 28 | d_model: 2048


In [ ]:
template = "{} + {} = {}"

def create_dataset(n_samples=500, seed=0):
    rng = random.Random(seed)
    data, seen = [], set()
    while len(data) < n_samples:
        ah, bh = rng.randint(1, 8), rng.randint(1, 8)
        if ah + bh > 9:
            continue
        at = rng.randint(0, 8)
        bt = rng.randint(0, 8 - at)
        ao, bo = rng.randint(0, 9), rng.randint(0, 9)
        if ao + bo >= 10:
            continue
        co = rng.randint(1, 9)
        do = rng.randint(10 - co, 9)
        if co + do < 10:
            continue
        a = 100 * ah + 10 * at + ao
        b = 100 * bh + 10 * bt + bo
        c = 100 * ah + 10 * at + co
        d = 100 * bh + 10 * bt + do
        base_sum, source_sum = a + b, c + d
        if base_sum // 100 != source_sum // 100 or not 0 <= base_sum // 100 <= 9:
            continue
        base_prompt = template.format(a, b, base_sum // 100)
        source_prompt = template.format(c, d, source_sum // 100)
        if len(base_prompt) != len(source_prompt):
            continue
        base_target = (base_sum // 10) % 10
        source_target = (source_sum // 10) % 10
        if source_target != base_target + 1:
            continue
        pair = (base_prompt, source_prompt, base_target, source_target)
        if pair not in seen:
            data.append(pair)
            seen.add(pair)
    return data

def _prompt_tokens(model, prompts, device):
    toks = [model.to_tokens(p)[0] for p in prompts]
    L = toks[0].shape[0]
    if any(t.shape[0] != L for t in toks):
        raise ValueError("All prompts must tokenize to the same length")
    return torch.stack(toks).to(device)

def get_digit_ids(model):
    ids = []
    for i in range(10):
        t = model.to_tokens(str(i))
        if t.shape[1] != 1:
            raise ValueError(f"digit {i} tokenizes to {t.shape[1]} tokens")
        ids.append(t[0, -1].item())
    return ids

def check_tokenization(model, prompt="410 + 432 = 8"):
    toks = list(model.to_str_tokens(prompt))
    expected = ['4', '1', '0', ' +', ' ', '4', '3', '2', ' =', ' ', '8']
    if toks != expected:
        raise ValueError(f"Unexpected tokenization {toks}; expected {expected}.")
    completed = list(model.to_str_tokens(prompt + "4"))
    if completed[:-1] != toks or completed[-1] != "4":
        raise ValueError("Tokenizer does not preserve prefix when digit appended.")
    return toks

@torch.no_grad()
def _score_next_digit(model, prompts, target_digits, digit_ids, batch_size=64, amp=True):
    dev = next(model.parameters()).device
    targets = torch.as_tensor([digit_ids[int(d)] for d in target_digits], device=dev, dtype=torch.long)
    preds, losses = [], []
    for s in range(0, len(prompts), batch_size):
        e = min(s + batch_size, len(prompts))
        toks = _prompt_tokens(model, prompts[s:e], dev)
        with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
            logits = model.run_with_hooks(toks, return_type="logits")[:, -1, :].float()
        preds.extend(logits.argmax(-1).cpu().tolist())
        losses.extend(F.cross_entropy(logits, targets[s:e], reduction="none").cpu().tolist())
    return preds, losses

def create_filtered_dataset(model, digit_ids, n_samples=300, seed=0, score_batch_size=64,
                            max_candidates_factor=20, amp=True):
    cand, accepted, seen = 0, [], set()
    bh_all, sh_all, bl_all, sl_all = [], [], [], []
    attempt = 0
    while len(accepted) < n_samples:
        remaining = n_samples - len(accepted)
        batch = create_dataset(max(32, math.ceil(1.28 * remaining)), seed=seed + attempt)
        attempt += 1
        fresh = []
        for row in batch:
            key = (row[0], row[1])
            if key not in seen:
                seen.add(key); fresh.append(row)
        if not fresh:
            continue
        bp, sp = [r[0] for r in fresh], [r[1] for r in fresh]
        bt, st = [r[2] for r in fresh], [r[3] for r in fresh]
        bpred, bloss = _score_next_digit(model, bp, bt, digit_ids, score_batch_size, amp)
        spred, sloss = _score_next_digit(model, sp, st, digit_ids, score_batch_size, amp)
        for i, row in enumerate(fresh):
            b_ok = bpred[i] == digit_ids[int(row[2])]
            s_ok = spred[i] == digit_ids[int(row[3])]
            cand += 1
            bh_all.append(b_ok); sh_all.append(s_ok); bl_all.append(bloss[i]); sl_all.append(sloss[i])
            if b_ok and s_ok and len(accepted) < n_samples:
                accepted.append((row, bloss[i], sloss[i]))
        if cand >= max_candidates_factor * n_samples and len(accepted) < n_samples:
            raise RuntimeError(f"Only {len(accepted)}/{n_samples} kept after {cand} candidates.")
    print(f"candidates={cand} | base acc={np.mean(bh_all):.3f} CE={np.mean(bl_all):.3f} | "
          f"source acc={np.mean(sh_all):.3f} CE={np.mean(sl_all):.3f} | "
          f"both-correct={sum(a and b for a, b in zip(bh_all, sh_all))}/{cand}")
    return [r for r, _, _ in accepted]

In [ ]:
#select nodes: skip first four tokens and skip last 20/30% layers
# NOTE: the ones digit of the 2nd number is at position 7, so the ones carry cannot exist before position 7
# (causal attention). Positions 4-6 are therefore never expected to be selected -- they act as a built-in
# negative control. Use range(7, 10) if you want a smaller, faster search.
nodes = []
for layer in range(int(0.6*model.cfg.n_layers)):
    for pos in range(5, 11):
        nodes.append((layer, pos))

max_layer = 17
final_prompt_pos = 11

nodes = [
    (layer, pos)
    for layer, pos in nodes
    if layer <= max_layer and pos != final_prompt_pos
]

print(f"Candidate nodes after filtering: {len(nodes)}")
print(nodes)

Candidate nodes after filtering: 96
[(0, 5), (0, 6), (0, 7), (0, 8), (0, 9), (0, 10), (1, 5), (1, 6), (1, 7), (1, 8), (1, 9), (1, 10), (2, 5), (2, 6), (2, 7), (2, 8), (2, 9), (2, 10), (3, 5), (3, 6), (3, 7), (3, 8), (3, 9), (3, 10), (4, 5), (4, 6), (4, 7), (4, 8), (4, 9), (4, 10), (5, 5), (5, 6), (5, 7), (5, 8), (5, 9), (5, 10), (6, 5), (6, 6), (6, 7), (6, 8), (6, 9), (6, 10), (7, 5), (7, 6), (7, 7), (7, 8), (7, 9), (7, 10), (8, 5), (8, 6), (8, 7), (8, 8), (8, 9), (8, 10), (9, 5), (9, 6), (9, 7), (9, 8), (9, 9), (9, 10), (10, 5), (10, 6), (10, 7), (10, 8), (10, 9), (10, 10), (11, 5), (11, 6), (11, 7), (11, 8), (11, 9), (11, 10), (12, 5), (12, 6), (12, 7), (12, 8), (12, 9), (12, 10), (13, 5), (13, 6), (13, 7), (13, 8), (13, 9), (13, 10), (14, 5), (14, 6), (14, 7), (14, 8), (14, 9), (14, 10), (15, 5), (15, 6), (15, 7), (15, 8), (15, 9), (15, 10)]


In [ ]:
def binary_concrete(logits, K=1.0, hard=True):
    soft = torch.sigmoid(K * logits)
    if not hard:
        return soft, soft
    hard_m = (logits > 0).to(logits.dtype)
    return hard_m - soft.detach() + soft, soft


class DASSubspaces(nn.Module):
    """Unconstrained W [n_nodes, d, k]; orthonormal basis Q via one batched QR per call."""
    def __init__(self, n_nodes, d_model, k=1):
        super().__init__()
        self.k = k
        self.W = nn.Parameter(torch.randn(n_nodes, d_model, k) / d_model ** 0.5)
    def basis(self):
        return torch.linalg.qr(self.W).Q          # [n, d, k], orthonormal columns


def prepare(model, data, nodes, gold_ids, cache_bs=64, amp=True):
    """Tokenize and cache SOURCE residual vectors for every node (kept on GPU, fp32)."""
    dev = next(model.parameters()).device
    gold = torch.as_tensor(gold_ids, dtype=torch.long, device=dev).view(-1)
    N = len(data)
    base_tok = _prompt_tokens(model, [r[0] for r in data], dev)
    src_tok = _prompt_tokens(model, [r[1] for r in data], dev)
    L = base_tok.shape[1]
    by_layer = {}
    for i, (layer, pos) in enumerate(nodes):
        by_layer.setdefault(layer, []).append((i, pos))
    layer_nodes = {l: torch.tensor([i for i, _ in v], device=dev) for l, v in by_layer.items()}
    layer_pos = {l: torch.tensor([p for _, p in v], device=dev) for l, v in by_layer.items()}

    src_cache = torch.zeros(len(nodes), N, model.cfg.d_model, device=dev)
    with torch.no_grad():
        for s in range(0, N, cache_bs):
            sl = slice(s, min(s + cache_bs, N))
            def make_hook(layer):
                def hook(act, hook=None):
                    src_cache[layer_nodes[layer], sl] = act[:, layer_pos[layer], :].float().transpose(0, 1)
                    return act
                return hook
            with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
                model.run_with_hooks(src_tok[sl],
                                     fwd_hooks=[(f"blocks.{l}.hook_resid_post", make_hook(l)) for l in by_layer],
                                     return_type=None)
    return dict(base_tok=base_tok, src_cache=src_cache, gold=gold, L=L, N=N,
                layers=sorted(by_layer), layer_nodes=layer_nodes, layer_pos=layer_pos)


def patched_logits(model, prep, idx, m, Q, amp=True):
    """Intervene (base <- source) in node subspaces, one fused op per layer. Returns [B, vocab]
    logits at the position predicting the tens digit."""
    def make_hook(layer):
        nid, pos = prep["layer_nodes"][layer], prep["layer_pos"][layer]
        def hook(act, hook=None):
            cur = act[:, pos, :].float()                                   # [B, n, d]
            src = prep["src_cache"][nid[:, None], idx[None, :]].transpose(0, 1)  # [B, n, d]
            q = Q[nid]                                                     # [n, d, k]
            coef = torch.einsum("bnd,ndk->bnk", src - cur, q)
            proj = torch.einsum("bnk,ndk->bnd", coef, q)
            new = cur + m[nid][None, :, None] * proj
            act = act.clone()
            act[:, pos, :] = new.to(act.dtype)
            return act
        return hook
    with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
        logits = model.run_with_hooks(prep["base_tok"][idx],
                                      fwd_hooks=[(f"blocks.{l}.hook_resid_post", make_hook(l)) for l in prep["layers"]],
                                      return_type="logits")
    return logits[:, prep["L"] - 1, :].float()


@torch.no_grad()
def evaluate_iia(model, prep, mask, Q, batch_size=100, amp=True):
    dev = prep["base_tok"].device
    m = torch.as_tensor(mask, device=dev).float()
    hits = []
    for s in range(0, prep["N"], batch_size):
        idx = torch.arange(s, min(s + batch_size, prep["N"]), device=dev)
        pred = patched_logits(model, prep, idx, m, Q, amp).argmax(-1)
        hits.append(pred == prep["gold"][idx])
    return torch.cat(hits).float().mean().item()


def random_mask_baseline(model, prep, n_nodes, k, Q, n_trials=20, amp=True):
    accs = []
    for _ in range(n_trials):
        m = torch.zeros(n_nodes); m[torch.randperm(n_nodes)[:k]] = 1
        accs.append(evaluate_iia(model, prep, m, Q, amp=amp))
    return float(np.mean(accs))


def leave_one_out(model, prep, mask, Q, nodes, amp=True):
    mask = torch.as_tensor(mask).clone()
    rows = []
    for i in torch.nonzero(mask).flatten().tolist():
        m = mask.clone(); m[i] = 0
        rows.append((nodes[i], evaluate_iia(model, prep, m, Q, amp=amp)))
    return rows


def summarize_nodes(sel):
    by_pos = {}
    for layer, pos in sel:
        by_pos.setdefault(pos, []).append(layer)
    lines = []
    for pos in sorted(by_pos):
        layers = sorted(by_pos[pos]); runs, start = [], layers[0]
        for a, b in zip(layers, layers[1:] + [None]):
            if b is None or b != a + 1:
                runs.append(f"{start}" if start == a else f"{start}-{a}"); start = b
        lines.append(f"  pos {pos}: layers {', '.join(runs)}")
    return "\n".join(lines) if lines else "  (no nodes selected)"


def alignment_find(model, prep, nodes, subspace_dim=1, init_logit=-0.5, lambda_=0.005, K=4.0,
                   lr_alpha=3e-2, lr_basis=1e-2, n_steps=300, batch_size=64, micro_batch=32,
                   hard=True, amp=True, seed=0, log_every=25):
    torch.manual_seed(seed)
    dev = prep["base_tok"].device
    N, n = prep["N"], len(nodes)
    sub = DASSubspaces(n, model.cfg.d_model, subspace_dim).to(dev)
    alpha = nn.Parameter(torch.full((n,), float(init_logit), device=dev))
    opt = torch.optim.Adam([{"params": [alpha], "lr": lr_alpha},
                            {"params": sub.parameters(), "lr": lr_basis}])
    t0 = time.time()
    pbar = tqdm(range(n_steps))
    for step in pbar:
        opt.zero_grad(set_to_none=True)
        perm = torch.randperm(N, device=dev)[:batch_size]
        mask_used, mask_soft = binary_concrete(alpha, K, hard)
        m_leaf = mask_used.detach().clone().requires_grad_(True)
        Q = sub.basis()
        Q_leaf = Q.detach().clone().requires_grad_(True)
        ce_total = 0.0
        for s in range(0, len(perm), micro_batch):
            idx = perm[s:s + micro_batch]
            logits = patched_logits(model, prep, idx, m_leaf, Q_leaf, amp)
            loss = F.cross_entropy(logits, prep["gold"][idx], reduction="sum") / len(perm)
            loss.backward()
            ce_total += loss.item()
        if not math.isfinite(ce_total):
            raise RuntimeError("Non-finite loss (fp16 overflow?). Re-run with amp=False.")
        Q.backward(Q_leaf.grad)                           # one backward through QR
        probs = torch.sigmoid(K * alpha)
        ((mask_used * m_leaf.grad).sum() + lambda_ * probs.sum()).backward()
        opt.step()
        if step == 2:
            print(f"~{(time.time() - t0) / 3:.1f}s/step -> ~{(time.time() - t0) / 3 * n_steps / 60:.1f} min total")
        if step % log_every == 0 or step == n_steps - 1:
            pbar.set_postfix(ce=f"{ce_total:.3f}", sel=int((alpha > 0).sum()))
    with torch.no_grad():
        hard_mask = (alpha > 0).long().cpu()
        Qf = sub.basis().detach()
    return dict(alpha=alpha.detach().cpu(), mask_hard=hard_mask, Q=Qf,
                selected_nodes=[nodes[i] for i in range(n) if hard_mask[i]])

In [ ]:
def run_carry_search(model, nodes, n_samples=300, seed=0, amp=True, do_loo=True,
                     n_random_trials=20, **search_kwargs):
    T = time.time()
    random.seed(seed); torch.manual_seed(seed)
    check_tokenization(model)
    digit_ids = get_digit_ids(model)
    data = create_filtered_dataset(model, digit_ids, n_samples=n_samples, seed=seed, amp=amp)
    gold_ids = [digit_ids[r[3]] for r in data]            # score against SOURCE tens digit
    prep = prepare(model, data, nodes, gold_ids, amp=amp)
    print(f"data+cache ready in {time.time() - T:.0f}s")

    Q0 = DASSubspaces(len(nodes), model.cfg.d_model, search_kwargs.get("subspace_dim", 1)).to(device).basis().detach()
    print("no-intervention acc:", evaluate_iia(model, prep, torch.zeros(len(nodes)), Q0, amp=amp))

    res = alignment_find(model, prep, nodes, amp=amp, seed=seed, **search_kwargs)
    k = int(res["mask_hard"].sum())
    print(f"selected {k}/{len(nodes)} nodes\n{summarize_nodes(res['selected_nodes'])}")
    if k == 0:
        alpha_np = res["alpha"].detach().cpu().numpy()
        prob_np = 1.0 / (1.0 + np.exp(-search_kwargs.get("K", 4.0) * alpha_np))
        top = np.argsort(alpha_np)[::-1][:min(10, len(nodes))]
        print("ZERO-NODE DIAGNOSTIC: no alpha crossed the hard threshold alpha > 0.")
        print(f"alpha range=[{alpha_np.min():.4f}, {alpha_np.max():.4f}], max inclusion probability={prob_np.max():.4f}")
        print("Top candidates (layer, position, alpha, soft inclusion probability):")
        for j in top:
            print(f"  {nodes[int(j)]}: alpha={alpha_np[j]:.4f}, p={prob_np[j]:.4f}")
        print("Treat this as a failed/empty discovery for these hyperparameters; do not invent a node or interpret it as proof that the feature is absent.")
    found = evaluate_iia(model, prep, res["mask_hard"], res["Q"], amp=amp)
    print(f"FOUND mask IIA (source tens acc, train pairs): {found:.3f}")
    rand = loo = None
    if k > 0:
        rand = random_mask_baseline(model, prep, len(nodes), k, res["Q"], n_random_trials, amp)
        print(f"RANDOM same-size mask IIA (mean of {n_random_trials}): {rand:.3f}")
        if do_loo and k <= 40:
            loo = leave_one_out(model, prep, res["mask_hard"], res["Q"], nodes, amp)
            for node, v in sorted(loo, key=lambda x: x[1]):
                print(f"  without {node}: {v:.3f}")
    print(f"done in {time.time() - T:.0f}s")
    return dict(result=res, found=found, random=rand, loo=loo, data=data, prep=prep)

In [ ]:
out = run_carry_search(
    model, nodes,
    n_samples=300,
    subspace_dim=64,
    lambda_=0.008, K=4.0,
    n_steps=300, batch_size=64, micro_batch=32,   # raise micro_batch to 64 if memory allows
    lr_alpha=3e-2, lr_basis=1e-2,
    amp=True,                                      # set False if you see non-finite loss
)
selected_nodes = out["result"]["selected_nodes"]
selected_nodes

candidates=384 | base acc=0.971 CE=0.148 | source acc=0.932 CE=0.320 | both-correct=354/384
data+cache ready in 6s
no-intervention acc: 0.0


  0%|          | 0/300 [00:00<?, ?it/s]

~1.2s/step -> ~5.9 min total
selected 3/96 nodes
  pos 8: layers 1-2
  pos 9: layers 10
FOUND mask IIA (source tens acc, train pairs): 0.803
RANDOM same-size mask IIA (mean of 20): 0.000
  without (10, 9): 0.000
  without (2, 8): 0.013
  without (1, 8): 0.017
done in 224s


[(1, 8), (2, 8), (10, 9)]

In [ ]:
out = run_carry_search(
    model, nodes,
    n_samples=300,
    subspace_dim=8,
    lambda_=0.01, K=4.0,
    n_steps=300, batch_size=64, micro_batch=32,   # raise micro_batch to 64 if memory allows
    lr_alpha=3e-2, lr_basis=1e-2,
    amp=True,                                      # set False if you see non-finite loss
)
selected_nodes = out["result"]["selected_nodes"]
selected_nodes

candidates=384 | base acc=0.971 CE=0.148 | source acc=0.932 CE=0.320 | both-correct=354/384
data+cache ready in 4s
no-intervention acc: 0.0


  0%|          | 0/300 [00:00<?, ?it/s]

~0.6s/step -> ~2.8 min total
selected 1/96 nodes
  pos 7: layers 6
FOUND mask IIA (source tens acc, train pairs): 0.730
RANDOM same-size mask IIA (mean of 20): 0.000
  without (6, 7): 0.000
done in 200s


[(6, 7)]

In [ ]:
out = run_carry_search(
    model, nodes,
    n_samples=300,
    subspace_dim=64,
    lambda_=0.008, K=4.0,
    n_steps=300, batch_size=64, micro_batch=32,   # raise micro_batch to 64 if memory allows
    lr_alpha=3e-2, lr_basis=1e-2,
    amp=True,                                      # set False if you see non-finite loss
)
selected_nodes = out["result"]["selected_nodes"]
selected_nodes

candidates=384 | base acc=0.971 CE=0.148 | source acc=0.932 CE=0.320 | both-correct=354/384
data+cache ready in 4s
no-intervention acc: 0.0


  0%|          | 0/300 [00:00<?, ?it/s]

~0.6s/step -> ~3.2 min total
selected 3/96 nodes
  pos 8: layers 1-2
  pos 9: layers 10
FOUND mask IIA (source tens acc, train pairs): 0.803
RANDOM same-size mask IIA (mean of 20): 0.000
  without (10, 9): 0.000
  without (2, 8): 0.013
  without (1, 8): 0.017
done in 223s


[(1, 8), (2, 8), (10, 9)]

In [ ]:
out = run_carry_search(
    model, nodes,
    n_samples=300,
    subspace_dim=64,
    lambda_=0.0095, K=4.0,
    n_steps=300, batch_size=64, micro_batch=32,   # raise micro_batch to 64 if memory allows
    lr_alpha=3e-2, lr_basis=1e-2,
    amp=True,                                      # set False if you see non-finite loss
)
selected_nodes = out["result"]["selected_nodes"]
selected_nodes

candidates=384 | base acc=0.971 CE=0.148 | source acc=0.932 CE=0.320 | both-correct=354/384
data+cache ready in 4s
no-intervention acc: 0.0


  0%|          | 0/300 [00:00<?, ?it/s]

~0.6s/step -> ~3.2 min total
selected 2/96 nodes
  pos 8: layers 1
  pos 9: layers 10
FOUND mask IIA (source tens acc, train pairs): 0.397
RANDOM same-size mask IIA (mean of 20): 0.000
  without (10, 9): 0.000
  without (1, 8): 0.007
done in 223s


[(1, 8), (10, 9)]

In [ ]:
out = run_carry_search(
    model, nodes,
    n_samples=300,
    subspace_dim=64,
    lambda_=0.01, K=4.0,
    n_steps=300, batch_size=64, micro_batch=32,   # raise micro_batch to 64 if memory allows
    lr_alpha=3e-2, lr_basis=1e-2,
    amp=True,                                      # set False if you see non-finite loss
)
selected_nodes = out["result"]["selected_nodes"]
selected_nodes

candidates=384 | base acc=0.971 CE=0.148 | source acc=0.932 CE=0.320 | both-correct=354/384
data+cache ready in 4s
no-intervention acc: 0.0


  0%|          | 0/300 [00:00<?, ?it/s]

~0.6s/step -> ~3.2 min total
selected 2/96 nodes
  pos 8: layers 1
  pos 9: layers 10
FOUND mask IIA (source tens acc, train pairs): 0.297
RANDOM same-size mask IIA (mean of 20): 0.000
  without (10, 9): 0.000
  without (1, 8): 0.010
done in 225s


[(1, 8), (10, 9)]

## Validation: freeze + refine, held-out pairs, control counterfactuals

Run `validate_search(model, out, nodes)` on the result of any search. It does five things with the selected nodes:

1. **Freeze + refine**: fixes the selected mask and retrains only the subspace bases `Q` (the search trained them against a moving mask).
2. **Held-out pairs**: fresh screened pairs (different seed, no overlap with training pairs). Frozen `Q` is scored on these.
3. **Reversed pairs** (base has carry, source does not): the tens digit should go *down* by 1 if `Q` carries a carry flag.
4. **Same-carry controls** (no-carry/no-carry and carry/carry, ones digits differ): the output must stay at the base answer. If `Q` moves *digit identity* rather than a carry flag, mixing digits can flip the carry here and accuracy drops below 1.
5. **Random directions at the same nodes**: same mask, random orthonormal `Q`. This should be ~0 on the main sets.

All numbers are accuracy against the intended answer for that set (see the row label).

In [ ]:
import pandas as pd


def screen_rows(model, rows, digit_ids, amp=True):
    """Keep only rows where the model gets BOTH base and source prompts right (row = (bp, sp, base_t, src_t))."""
    bpred, _ = _score_next_digit(model, [r[0] for r in rows], [r[2] for r in rows], digit_ids, 64, amp)
    spred, _ = _score_next_digit(model, [r[1] for r in rows], [r[3] for r in rows], digit_ids, 64, amp)
    return [r for r, b, s in zip(rows, bpred, spred)
            if b == digit_ids[r[2]] and s == digit_ids[r[3]]]


def make_same_carry_pairs(n, carry, seed):
    """Pairs with identical hundreds/tens digits, DIFFERENT ones digits, and the SAME ones-carry status.
    carry=False: both prompts have no ones carry. carry=True: both have a ones carry.
    The intended output after patching is the BASE tens digit (== source tens digit)."""
    rng = random.Random(seed)
    rows, seen = [], set()

    def ones():
        while True:
            x, y = rng.randint(0, 9), rng.randint(0, 9)
            if (x + y >= 10) == carry:
                return x, y

    while len(rows) < n:
        ah, bh = rng.randint(1, 8), rng.randint(1, 8)
        if ah + bh > 9:
            continue
        at = rng.randint(0, 8)
        bt = rng.randint(0, 8 - at)          # tens sum <= 8 -> no tens carry even with the ones carry
        (ao, bo), (co, do) = ones(), ones()
        if (ao, bo) == (co, do):
            continue
        a, b = 100 * ah + 10 * at + ao, 100 * bh + 10 * bt + bo
        c, d = 100 * ah + 10 * at + co, 100 * bh + 10 * bt + do
        bs, ss = a + b, c + d
        if bs // 100 != ss // 100 or (bs // 10) % 10 != (ss // 10) % 10:
            continue
        bp = template.format(a, b, bs // 100)
        sp = template.format(c, d, ss // 100)
        if len(bp) != len(sp) or (bp, sp) in seen:
            continue
        seen.add((bp, sp))
        t = (bs // 10) % 10
        rows.append((bp, sp, t, t))
    return rows


def refine_Q(model, prep, Q_init, n_steps=100, lr=2e-3, batch_size=64, micro_batch=32, amp=True, seed=0):
    """Mask frozen at all-ones over the selected nodes; retrain only the bases (one QR backward per step)."""
    torch.manual_seed(seed)
    dev = prep["base_tok"].device
    W = nn.Parameter(Q_init.detach().clone().to(dev))
    opt = torch.optim.Adam([W], lr=lr)
    m = torch.ones(W.shape[0], device=dev)
    for _ in tqdm(range(n_steps), desc="refine Q"):
        opt.zero_grad(set_to_none=True)
        perm = torch.randperm(prep["N"], device=dev)[:batch_size]
        Q = torch.linalg.qr(W).Q
        Q_leaf = Q.detach().clone().requires_grad_(True)
        for s in range(0, len(perm), micro_batch):
            idx = perm[s:s + micro_batch]
            logits = patched_logits(model, prep, idx, m, Q_leaf, amp)
            loss = F.cross_entropy(logits, prep["gold"][idx], reduction="sum") / len(perm)
            loss.backward()
        Q.backward(Q_leaf.grad)
        opt.step()
    return torch.linalg.qr(W).Q.detach()


def acc_on_rows(model, rows, sel_nodes, Q, digit_ids, amp=True):
    """Patch ALL selected nodes (mask=1) with basis Q; accuracy vs the row's intended digit (row[3])."""
    gold = [digit_ids[r[3]] for r in rows]
    prep = prepare(model, rows, sel_nodes, gold, amp=amp)
    return evaluate_iia(model, prep, torch.ones(len(sel_nodes)), Q, amp=amp)


def validate_search(model, out, nodes, n_heldout=300, n_controls=200, refine_steps=100,
                    refine_lr=2e-3, seed=1000, amp=True):
    T = time.time()
    dev = next(model.parameters()).device
    digit_ids = get_digit_ids(model)
    res, train = out["result"], out["data"]
    sel = torch.nonzero(res["mask_hard"]).flatten().tolist()
    if not sel:
        raise ValueError("No nodes selected in this search; nothing to validate.")
    sel_nodes = [nodes[i] for i in sel]
    Q0 = res["Q"][sel].to(dev)
    print(f"validating {len(sel_nodes)} nodes: {sel_nodes}  (subspace dim {Q0.shape[-1]})")

    # ---- datasets -----------------------------------------------------------------------------
    train_keys = {(r[0], r[1]) for r in train}
    ho = create_filtered_dataset(model, digit_ids, n_samples=n_heldout + 60, seed=seed, amp=amp)
    ho = [r for r in ho if (r[0], r[1]) not in train_keys][:n_heldout]
    # reversed: base <- carry prompt, source <- no-carry prompt; intended = original base tens digit
    rev = [(r[1], r[0], r[3], r[2]) for r in ho]
    nn_rows = screen_rows(model, make_same_carry_pairs(int(n_controls * 1.5), False, seed + 1), digit_ids, amp)[:n_controls]
    cc_rows = screen_rows(model, make_same_carry_pairs(int(n_controls * 1.5), True, seed + 2), digit_ids, amp)[:n_controls]
    sets = [
        ("train pairs: no-carry <- carry (expect source = base+1)", train),
        ("HELD-OUT pairs: no-carry <- carry (expect base+1)", ho),
        ("reversed: carry <- no-carry (expect base-1)", rev),
        ("control: no-carry <- no-carry (expect UNCHANGED)", nn_rows),
        ("control: carry <- carry (expect UNCHANGED)", cc_rows),
    ]
    print("set sizes:", [len(s[1]) for s in sets])

    # ---- refine on train ----------------------------------------------------------------------
    gold_tr = [digit_ids[r[3]] for r in train]
    prep_tr = prepare(model, train, sel_nodes, gold_tr, amp=amp)
    Q1 = refine_Q(model, prep_tr, Q0, n_steps=refine_steps, lr=refine_lr, amp=amp)
    Qr = torch.linalg.qr(torch.randn_like(Q0)).Q          # random directions, same nodes

    # ---- evaluate ------------------------------------------------------------------------------
    rows = []
    for name, data in sets:
        rows.append({"set": name, "n": len(data),
                     "search Q": acc_on_rows(model, data, sel_nodes, Q0, digit_ids, amp),
                     "refined Q": acc_on_rows(model, data, sel_nodes, Q1, digit_ids, amp),
                     "random Q": acc_on_rows(model, data, sel_nodes, Qr, digit_ids, amp)})
    df = pd.DataFrame(rows).set_index("set")
    print(df.round(3).to_string())
    print(f"done in {time.time() - T:.0f}s")
    return dict(table=df, sel_nodes=sel_nodes, Q_search=Q0, Q_refined=Q1, heldout=ho)

### How to read the table
- **Held-out ≈ train** → the subspaces generalise. Held-out far below train → overfitting (typical at `subspace_dim=64` with 300 pairs).
- **Refined > search** → the search left accuracy on the table because bases were co-adapting to a moving mask. Use the refined number.
- **Reversed ≈ high** → the subspace behaves like a carry *flag* that can be written in both directions.
- **Controls near 1.0** → patching does not break same-carry cases. Controls well below 1.0 → the subspace carries ones-digit *identity* (or something else carry-independent) and the mixed digits flip the carry.
- **Random Q near 0 on the main sets** → the learned directions matter, not just the node choice.

In [ ]:
# Re-run the dim-8 search (this reproduced the single-node (6, 7) solution) and validate it.
out8 = run_carry_search(
    model, nodes,
    n_samples=300,
    subspace_dim=8,
    lambda_=0.01, K=4.0,
    n_steps=300, batch_size=64, micro_batch=32,
    lr_alpha=3e-2, lr_basis=1e-2,
    amp=True,
)
report8 = validate_search(model, out8, nodes)

candidates=384 | base acc=0.971 CE=0.148 | source acc=0.932 CE=0.320 | both-correct=354/384
data+cache ready in 5s
no-intervention acc: 0.0


  0%|          | 0/300 [00:00<?, ?it/s]

~1.5s/step -> ~7.4 min total
selected 1/96 nodes
  pos 7: layers 6
FOUND mask IIA (source tens acc, train pairs): 0.730
RANDOM same-size mask IIA (mean of 20): 0.000
  without (6, 7): 0.000
done in 186s
validating 1 nodes: [(6, 7)]  (subspace dim 8)
candidates=461 | base acc=0.967 CE=0.207 | source acc=0.939 CE=0.334 | both-correct=429/461
set sizes: [300, 300, 300, 200, 200]


refine Q:   0%|          | 0/100 [00:00<?, ?it/s]

                                                           n  search Q  refined Q  random Q
set                                                                                        
train pairs: no-carry <- carry (expect source = base+1)  300     0.730      0.810     0.000
HELD-OUT pairs: no-carry <- carry (expect base+1)        300     0.463      0.507     0.000
reversed: carry <- no-carry (expect base-1)              300     0.393      0.490     0.000
control: no-carry <- no-carry (expect UNCHANGED)         200     0.855      0.820     0.995
control: carry <- carry (expect UNCHANGED)               200     0.900      0.865     1.000
done in 81s


In [ ]:
# Optional: validate the 3-node dim-64 solution the same way (re-run the search first so `out` matches).
out64 = run_carry_search(
    model, nodes,
    n_samples=300,
    subspace_dim=64,
    lambda_=0.008, K=4.0,
    n_steps=300, batch_size=64, micro_batch=32,
    lr_alpha=3e-2, lr_basis=1e-2,
    amp=True,
)
report64 = validate_search(model, out64, nodes, refine_lr=1e-3)

candidates=384 | base acc=0.971 CE=0.148 | source acc=0.932 CE=0.320 | both-correct=354/384
data+cache ready in 4s
no-intervention acc: 0.0


  0%|          | 0/300 [00:00<?, ?it/s]

~0.6s/step -> ~2.9 min total
selected 3/96 nodes
  pos 8: layers 1-2
  pos 9: layers 10
FOUND mask IIA (source tens acc, train pairs): 0.803
RANDOM same-size mask IIA (mean of 20): 0.000
  without (10, 9): 0.000
  without (2, 8): 0.013
  without (1, 8): 0.017
done in 206s
validating 3 nodes: [(1, 8), (2, 8), (10, 9)]  (subspace dim 64)
candidates=461 | base acc=0.967 CE=0.207 | source acc=0.939 CE=0.334 | both-correct=429/461
set sizes: [300, 300, 300, 200, 200]


refine Q:   0%|          | 0/100 [00:00<?, ?it/s]

                                                           n  search Q  refined Q  random Q
set                                                                                        
train pairs: no-carry <- carry (expect source = base+1)  300     0.800      0.837       0.0
HELD-OUT pairs: no-carry <- carry (expect base+1)        300     0.423      0.430       0.0
reversed: carry <- no-carry (expect base-1)              300     0.003      0.003       0.0
control: no-carry <- no-carry (expect UNCHANGED)         200     0.835      0.795       1.0
control: carry <- carry (expect UNCHANGED)               200     0.945      0.930       1.0
done in 84s


# Rigorous follow-up: confirmatory DAS evaluation

The earlier results are **exploratory**: the dim-8 solution achieved 50.7% on the reported held-out set, while the dim-64 solution achieved 43.0%; the reversed test and same-carry controls also reveal limitations. Do not interpret selected nodes as a validated causal abstraction yet.

This section adds a pre-specified evaluation protocol and reusable code. It is intentionally **not auto-run**: a full multi-seed sweep and locked test can take substantial GPU time. Run the search/selection experiments first, choose the candidate using development data only, and then run the locked final test once.

## Experiments and acceptance criteria

1. **Repeated-seed discovery and stability.** Run at least 5 independent search seeds for dimensions 1, 4, 8, and 16 with the same training examples across dimensions within each seed. Report selected-node frequency and pairwise Jaccard overlap. If discovered nodes vary greatly across seeds, do not claim a stable localization.
2. **Capacity sweep.** Compare dimensions 1/4/8/16 (optionally 32/64) while holding seeds, node candidates, optimizer steps, and data fixed. Report train, development, and locked test metrics. Do not choose the dimension using the locked test.
3. **Counterfactual tests.** Evaluate (a) no-carry → carry, (b) reversed carry → no-carry, (c) no-carry → no-carry, and (d) carry → carry. The same-carry controls should remain unchanged; forward and reverse interventions should both work if the subspace behaves like a manipulable carry variable.
4. **Intervention baselines and causal ablations.** Compare learned subspaces against no intervention, random orthonormal directions at the same discovered nodes, and a full-residual patch at those nodes. On held-out data, ablate each selected node one at a time while keeping the learned bases fixed. The full patch is an upper-bound-style diagnostic, not a fair low-dimensional competitor.
5. **Negative-control locations.** Search only positions 5–6 (before the ones-digit token at position 7 under the checked tokenizer). A comparable result from these positions would weaken the causal timing interpretation. This is an additional optimization run, not a free baseline.
6. **Uncertainty and paired comparison.** Report Wilson 95% binomial intervals and paired bootstrap intervals for accuracy differences on the exact same examples. Treat each prompt pair—not each token—as the statistical unit.
7. **Screening caveat.** The existing data pipeline screens for examples where the model predicts both the base and source prompts correctly. Conclusions therefore apply to this screened population. Also report the candidate acceptance rate; do not imply results apply to all addition prompts.

Suggested success criteria to define *before* the final test: high held-out forward accuracy with a narrow confidence interval; successful reverse transfer; both same-carry controls near baseline/ceiling; improvement over no-intervention and random-direction controls; repeatable node localization; and no comparable signal from pre-carry positions. These are criteria to test, not results already established.


In [ ]:
# Statistical helpers: 95% Wilson interval and paired bootstrap for accuracy differences.
import numpy as np
import pandas as pd
import math

def wilson_ci(correct, n, z=1.959963984540054):
    if n <= 0:
        return (float("nan"), float("nan"))
    p = correct / n
    den = 1 + z*z/n
    center = (p + z*z/(2*n)) / den
    half = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / den
    return max(0.0, center-half), min(1.0, center+half)

def summarize_hits(hits):
    hits = np.asarray(hits, dtype=bool).reshape(-1)
    n = len(hits)
    correct = int(hits.sum())
    lo, hi = wilson_ci(correct, n)
    return {"n": n, "correct": correct, "accuracy": correct/n if n else float("nan"),
            "ci95_low": lo, "ci95_high": hi}

def paired_bootstrap_accuracy_diff(hits_a, hits_b, n_boot=5000, seed=0):
    """Percentile CI for acc(A)-acc(B), paired by identical examples."""
    a, b = np.asarray(hits_a, dtype=bool), np.asarray(hits_b, dtype=bool)
    if a.shape != b.shape or a.size == 0:
        raise ValueError("Paired hit arrays must be non-empty and have identical shapes")
    diff = a.astype(np.float64) - b.astype(np.float64)
    rng = np.random.default_rng(seed)
    means = np.empty(n_boot, dtype=np.float64)
    for i in range(n_boot):
        means[i] = diff[rng.integers(0, len(diff), size=len(diff))].mean()
    return {"difference": float(diff.mean()),
            "ci95_low": float(np.quantile(means, 0.025)),
            "ci95_high": float(np.quantile(means, 0.975))}

print("Statistical helpers ready.")


Statistical helpers ready.


In [ ]:
def _collect_disjoint_filtered(model, digit_ids, n_samples, seed, blocked_prompts,
                               amp=True, max_rounds=30):
    """Collect screened no-carry -> carry pairs with no prompt shared with blocked data.

    Rows preserve the notebook schema (base_prompt, source_prompt, base_tens, source_tens).
    This blocks individual prompts, not only identical pairs, across train/final evaluation.
    """
    used = set(blocked_prompts)
    accepted, round_id = [], 0
    while len(accepted) < n_samples and round_id < max_rounds:
        need = n_samples - len(accepted)
        batch_n = max(64, 2*need)
        batch = create_filtered_dataset(model, digit_ids, n_samples=batch_n,
                                        seed=seed + 7919*round_id, amp=amp)
        for row in batch:
            bp, sp = row[0], row[1]
            if bp in used or sp in used or bp == sp:
                continue
            accepted.append(row)
            used.add(bp); used.add(sp)
            if len(accepted) >= n_samples:
                break
        round_id += 1
    if len(accepted) < n_samples:
        raise RuntimeError(f"Only collected {len(accepted)}/{n_samples} disjoint screened pairs. "
                           "Increase max_rounds or reduce the requested set size.")
    return accepted, used

def _collect_disjoint_controls(model, digit_ids, n_samples, carry, seed,
                               blocked_prompts, amp=True, max_rounds=50):
    """Collect screened same-carry controls without reusing prompts across splits."""
    used = set(blocked_prompts)
    accepted, round_id = [], 0
    while len(accepted) < n_samples and round_id < max_rounds:
        need = n_samples - len(accepted)
        raw = make_same_carry_pairs(max(80, 4*need), bool(carry), seed + 1009*round_id)
        screened = screen_rows(model, raw, digit_ids, amp=amp)
        for row in screened:
            bp, sp = row[0], row[1]
            if bp in used or sp in used or bp == sp:
                continue
            accepted.append(row)
            used.add(bp); used.add(sp)
            if len(accepted) >= n_samples:
                break
        round_id += 1
    if len(accepted) < n_samples:
        raise RuntimeError(f"Only collected {len(accepted)}/{n_samples} disjoint control pairs. "
                           "Increase max_rounds or reduce the requested set size.")
    return accepted, used

def build_locked_benchmark(model, train_rows, n_forward=300, n_controls=200,
                           seed=42000, amp=True):
    """Create one locked test benchmark disjoint at the individual-prompt level from training.

    Call this only after model selection is finished. Keep the returned benchmark fixed and do not
    change hyperparameters or selected nodes based on its scores.
    """
    digit_ids = get_digit_ids(model)
    blocked = {p for row in train_rows for p in row[:2]}
    forward, blocked = _collect_disjoint_filtered(
        model, digit_ids, n_forward, seed, blocked, amp=amp)
    # The reversed set uses the same underlying pairs by design: it tests reverse directionality.
    reverse = [(r[1], r[0], r[3], r[2]) for r in forward]
    no_carry, blocked = _collect_disjoint_controls(
        model, digit_ids, n_controls, False, seed+1, blocked, amp=amp)
    carry, blocked = _collect_disjoint_controls(
        model, digit_ids, n_controls, True, seed+2, blocked, amp=amp)
    out = {"forward": forward, "reverse": reverse,
           "control_no_carry": no_carry, "control_carry": carry,
           "digit_ids": digit_ids, "seed": seed}
    sizes = {k: len(v) for k, v in out.items() if isinstance(v, list)}
    print("Locked benchmark pair counts:", sizes)
    print("Individual prompts reused across train/test:", 0)
    print("Note: forward/reverse sets intentionally reuse the same pair in opposite directions.")
    return out

print("Disjoint benchmark builders ready. Build the locked benchmark only after fixing the candidate.")


Disjoint benchmark builders ready. Build the locked benchmark only after fixing the candidate.


In [ ]:
@torch.no_grad()
def _predict_hits_prepared(model, prep, Q, mask, amp=True, batch_size=64):
    dev = prep["base_tok"].device
    mask = torch.as_tensor(mask, dtype=torch.float32, device=dev)
    hit_chunks = []
    for start in range(0, prep["N"], batch_size):
        idx = torch.arange(start, min(start+batch_size, prep["N"]), device=dev)
        logits = patched_logits(model, prep, idx, mask, Q, amp=amp)
        hit_chunks.append((logits.argmax(-1) == prep["gold"][idx]).cpu())
    return torch.cat(hit_chunks).numpy().astype(bool)

@torch.no_grad()
def _predict_hits_full_resid_prepared(model, prep, mask, amp=True, batch_size=64):
    """Efficient exact residual patch at selected nodes without multiplying by an identity basis."""
    dev = prep["base_tok"].device
    mask = torch.as_tensor(mask, dtype=torch.float32, device=dev)
    hit_chunks = []
    for start in range(0, prep["N"], batch_size):
        idx = torch.arange(start, min(start+batch_size, prep["N"]), device=dev)
        def make_hook(layer):
            nid, pos = prep["layer_nodes"][layer], prep["layer_pos"][layer]
            def hook(act, hook=None):
                cur = act[:, pos, :].float()
                src = prep["src_cache"][nid[:, None], idx[None, :]].transpose(0, 1)
                new = cur + mask[nid][None, :, None] * (src-cur)
                act = act.clone()
                act[:, pos, :] = new.to(act.dtype)
                return act
            return hook
        with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
            logits = model.run_with_hooks(
                prep["base_tok"][idx],
                fwd_hooks=[(f"blocks.{layer}.hook_resid_post", make_hook(layer))
                           for layer in prep["layers"]],
                return_type="logits")[:, prep["L"]-1, :].float()
        hit_chunks.append((logits.argmax(-1) == prep["gold"][idx]).cpu())
    return torch.cat(hit_chunks).numpy().astype(bool)

def _prepare_eval_rows(model, rows, sel_nodes, digit_ids, amp=True):
    if not rows:
        raise ValueError("Evaluation set is empty")
    gold_ids = [digit_ids[int(r[3])] for r in rows]
    return prepare(model, rows, sel_nodes, gold_ids, amp=amp)

def score_locked_benchmark(model, benchmark, sel_nodes, Q, amp=True,
                           random_trials=10, include_full_patch=True, bootstrap_seed=0):
    """Score a frozen candidate on all locked sets with uncertainty and paired differences.

    Q must already be frozen/refined using training data only. Every method is evaluated on the
    same examples within a set, enabling paired bootstrap comparisons.
    """
    dev = next(model.parameters()).device
    k_nodes, d_model, subspace_dim = Q.shape
    learned_mask = torch.ones(k_nodes, device=dev)
    zero_mask = torch.zeros(k_nodes, device=dev)
    Q = Q.detach().to(dev)
    Q_random_list = [torch.linalg.qr(torch.randn_like(Q)).Q for _ in range(random_trials)]

    summary_rows, detailed_hits = [], {}
    set_labels = {
        "forward": "no-carry -> carry (expect source tens digit)",
        "reverse": "carry -> no-carry (expect original base tens digit)",
        "control_no_carry": "no-carry -> no-carry (expect unchanged digit)",
        "control_carry": "carry -> carry (expect unchanged digit)",
    }
    for set_key, label in set_labels.items():
        rows = benchmark[set_key]
        prep = _prepare_eval_rows(model, rows, sel_nodes, benchmark["digit_ids"], amp=amp)
        methods = {
            "no_intervention": _predict_hits_prepared(model, prep, Q, zero_mask, amp=amp),
            "learned_subspace": _predict_hits_prepared(model, prep, Q, learned_mask, amp=amp),
        }
        random_hits = [_predict_hits_prepared(model, prep, q, learned_mask, amp=amp)
                       for q in Q_random_list]
        # Held-out leave-one-node-out ablation: keep Q fixed and disable one selected node.
        for node_idx, node in enumerate(sel_nodes):
            ablated_mask = learned_mask.clone()
            ablated_mask[node_idx] = 0.0
            methods[f"without_node_{node}"] = _predict_hits_prepared(
                model, prep, Q, ablated_mask, amp=amp)
        if include_full_patch:
            methods["full_residual_patch"] = _predict_hits_full_resid_prepared(
                model, prep, learned_mask, amp=amp)
        detailed_hits[set_key] = dict(methods)
        learned_stats = summarize_hits(methods["learned_subspace"])
        summary_rows.append({"set": label, "method": "learned_subspace", **learned_stats})
        baseline_methods = ["no_intervention"] + (["full_residual_patch"] if include_full_patch else [])
        baseline_methods += [k for k in methods if k.startswith("without_node_")]
        for method in baseline_methods:
            summary_rows.append({"set": label, "method": method, **summarize_hits(methods[method])})
        r_acc = np.array([h.mean() for h in random_hits], dtype=np.float64)
        summary_rows.append({"set": label, "method": "random_subspace (mean over trials)",
                             "n": len(rows), "correct": np.nan, "accuracy": float(r_acc.mean()),
                             "ci95_low": float(np.quantile(r_acc, 0.025)),
                             "ci95_high": float(np.quantile(r_acc, 0.975)),
                             "note": "interval is across random-direction trials, not prompt-level CI"})
        for method in baseline_methods:
            delta = paired_bootstrap_accuracy_diff(methods["learned_subspace"], methods[method],
                                                   seed=bootstrap_seed)
            print(f"{set_key}: learned - {method}: {delta['difference']:+.3f} "
                  f"(paired bootstrap 95% CI {delta['ci95_low']:+.3f}, {delta['ci95_high']:+.3f})")
    df = pd.DataFrame(summary_rows)
    return {"table": df, "hits": detailed_hits, "sel_nodes": list(sel_nodes),
            "subspace_dim": subspace_dim, "n_nodes": k_nodes}

print("Frozen-candidate evaluation ready.")


Frozen-candidate evaluation ready.


## Running the experiments

### A. Confirmatory locked test (run once after selection)

First select a candidate using the development data. Then create a locked benchmark disjoint from the training prompts and evaluate the frozen/refined subspace. This is the final confirmatory test; do not use the scores to change the candidate and call the same test a final test again.

The benchmark builder is deliberately separate from candidate fitting. For strict model-selection practice, use a separate development set for choosing `subspace_dim`, `lambda_`, and node-search constraints before building the locked benchmark.

### B. Multi-seed/dimension stability sweep

The next cell is opt-in. Five seeds × four dimensions is the recommended run; it is expensive. Use the same seed across dimensions so that the screened training set is the same within each seed. Development validation must not be treated as the locked final test.


In [ ]:
def run_discovery_sweep(model, nodes, seeds=(0, 1, 2, 3, 4),
                        dimensions=(1, 4, 8, 16), n_samples=300,
                        n_steps=300, lambda_=0.0095, K=4.0, amp=True,
                        n_heldout=300, n_controls=200, refine_steps=100,
                        batch_size=64, micro_batch=32):
    """Run repeated-seed discovery; returns detailed rows and node-selection stability.

    For each seed, dimensions share the same deterministic training examples. The validation
    benchmark also uses a fixed seed per search seed, so dimensions within a seed are comparable.
    This is a development sweep; reserve a separate locked benchmark for final reporting.
    """
    records, selected = [], {}
    for seed in seeds:
        for dim in dimensions:
            print(f"\n===== discovery seed={seed}, subspace_dim={dim} =====")
            out = run_carry_search(
                model, nodes, n_samples=n_samples, seed=seed, amp=amp,
                subspace_dim=dim, lambda_=lambda_, K=K, n_steps=n_steps,
                batch_size=batch_size, micro_batch=micro_batch,
                lr_alpha=3e-2, lr_basis=1e-2)
            selected[(seed, dim)] = tuple(sorted(tuple(x) for x in out["result"]["selected_nodes"]))
            if not selected[(seed, dim)]:
                # Empty masks are valid sweep outcomes (especially at small dimensions / high sparsity).
                # Record them and continue; validation requires at least one selected node.
                print("ZERO-NODE RUN: skipping validation for this setting and continuing the sweep.")
                records.append({
                    "seed": seed, "subspace_dim": dim, "n_selected_nodes": 0,
                    "selected_nodes": (), "train_search_acc": out["found"],
                    "heldout_search_Q": np.nan, "heldout_refined_Q": np.nan,
                    "reverse_refined_Q": np.nan, "no_carry_control_refined_Q": np.nan,
                    "carry_control_refined_Q": np.nan, "status": "no_nodes_selected",
                })
                continue
            report = validate_search(
                model, out, nodes, n_heldout=n_heldout, n_controls=n_controls,
                refine_steps=refine_steps, seed=10000+seed, amp=amp)
            tab = report["table"]
            fwd_key = "HELD-OUT pairs: no-carry <- carry (expect base+1)"
            rev_key = "reversed: carry <- no-carry (expect base-1)"
            nn_key = "control: no-carry <- no-carry (expect UNCHANGED)"
            cc_key = "control: carry <- carry (expect UNCHANGED)"
            records.append({
                "seed": seed, "subspace_dim": dim,
                "n_selected_nodes": len(selected[(seed, dim)]),
                "selected_nodes": selected[(seed, dim)],
                "train_search_acc": out["found"],
                "heldout_search_Q": float(tab.loc[fwd_key, "search Q"]),
                "heldout_refined_Q": float(tab.loc[fwd_key, "refined Q"]),
                "reverse_refined_Q": float(tab.loc[rev_key, "refined Q"]),
                "no_carry_control_refined_Q": float(tab.loc[nn_key, "refined Q"]),
                "carry_control_refined_Q": float(tab.loc[cc_key, "refined Q"]),
                "status": "validated",
            })
    detail = pd.DataFrame(records)
    aggregate = detail.groupby("subspace_dim").agg(
        runs=("seed", "count"),
        zero_node_runs=("n_selected_nodes", lambda x: int((x == 0).sum())),
        train_mean=("train_search_acc", "mean"), train_sd=("train_search_acc", "std"),
        heldout_mean=("heldout_refined_Q", "mean"), heldout_sd=("heldout_refined_Q", "std"),
        reverse_mean=("reverse_refined_Q", "mean"), reverse_sd=("reverse_refined_Q", "std"),
        no_carry_control_mean=("no_carry_control_refined_Q", "mean"),
        carry_control_mean=("carry_control_refined_Q", "mean"),
    ).reset_index()
    freq_rows = []
    total_runs = len(selected)
    all_nodes = sorted({node for val in selected.values() for node in val})
    for node in all_nodes:
        count = sum(node in val for val in selected.values())
        freq_rows.append({"node": node, "selected_count": count,
                          "selection_frequency": count/total_runs if total_runs else np.nan})
    frequency = pd.DataFrame(freq_rows).sort_values("selection_frequency", ascending=False) if freq_rows else pd.DataFrame()
    # Pairwise Jaccard overlap of discovered node sets, across all runs.
    jaccard_rows = []
    keys = list(selected)
    for i in range(len(keys)):
        for j in range(i+1, len(keys)):
            a, b = set(selected[keys[i]]), set(selected[keys[j]])
            union = a | b
            jaccard_rows.append({"run_a": keys[i], "run_b": keys[j],
                                 "jaccard": len(a & b)/len(union) if union else np.nan,
                                 "both_empty": not a and not b})
    jaccard = pd.DataFrame(jaccard_rows)
    return {"detail": detail, "aggregate": aggregate,
            "node_frequency": frequency, "pairwise_jaccard": jaccard,
            "selected_by_run": selected}

# Explicit opt-in prevents a multi-hour GPU sweep from running accidentally.
RUN_DISCOVERY_SWEEP = True
if RUN_DISCOVERY_SWEEP:
    sweep = run_discovery_sweep(model, nodes, seeds=(0, 1, 2, 3, 4),
                                dimensions=(1, 4, 8, 16), n_samples=300,
                                n_steps=300, n_heldout=300, n_controls=200,
                                refine_steps=100, amp=True)
    display(sweep["aggregate"])
    display(sweep["node_frequency"])
    display(sweep["pairwise_jaccard"].describe())
else:
    print("Sweep skipped by default because it can be expensive. Set RUN_DISCOVERY_SWEEP=True to run it; zero-node settings will be recorded and the sweep will continue.")



===== discovery seed=0, subspace_dim=1 =====
candidates=384 | base acc=0.971 CE=0.148 | source acc=0.932 CE=0.320 | both-correct=354/384
data+cache ready in 4s
no-intervention acc: 0.0


  0%|          | 0/300 [00:00<?, ?it/s]

~0.5s/step -> ~2.6 min total
selected 0/96 nodes
  (no nodes selected)
ZERO-NODE DIAGNOSTIC: no alpha crossed the hard threshold alpha > 0.
alpha range=[-1.9216, -1.4376], max inclusion probability=0.0032
Top candidates (layer, position, alpha, soft inclusion probability):
  (12, 7): alpha=-1.4376, p=0.0032
  (7, 7): alpha=-1.7939, p=0.0008
  (11, 7): alpha=-1.7972, p=0.0008
  (5, 7): alpha=-1.8150, p=0.0007
  (1, 7): alpha=-1.8332, p=0.0007
  (2, 7): alpha=-1.8638, p=0.0006
  (10, 7): alpha=-1.8668, p=0.0006
  (9, 7): alpha=-1.8711, p=0.0006
  (6, 7): alpha=-1.8715, p=0.0006
  (11, 8): alpha=-1.8789, p=0.0005
Treat this as a failed/empty discovery for these hyperparameters; do not invent a node or interpret it as proof that the feature is absent.
FOUND mask IIA (source tens acc, train pairs): 0.000
done in 159s
ZERO-NODE RUN: skipping validation for this setting and continuing the sweep.

===== discovery seed=0, subspace_dim=4 =====
candidates=384 | base acc=0.971 CE=0.148 | source ac

  0%|          | 0/300 [00:00<?, ?it/s]

~0.5s/step -> ~2.6 min total
selected 1/96 nodes
  pos 7: layers 10
FOUND mask IIA (source tens acc, train pairs): 0.697
RANDOM same-size mask IIA (mean of 20): 0.000
  without (10, 7): 0.000
done in 179s
validating 1 nodes: [(10, 7)]  (subspace dim 4)
candidates=461 | base acc=0.967 CE=0.187 | source acc=0.961 CE=0.229 | both-correct=434/461
set sizes: [300, 300, 300, 200, 200]


...

refine Q:   0%|          | 0/100 [00:00<?, ?it/s]

                                                           n  search Q  refined Q  random Q
set                                                                                        
train pairs: no-carry <- carry (expect source = base+1)  300     0.920      0.930     0.000
HELD-OUT pairs: no-carry <- carry (expect base+1)        300     0.723      0.740     0.000
reversed: carry <- no-carry (expect base-1)              300     0.270      0.310     0.000
control: no-carry <- no-carry (expect UNCHANGED)         200     0.735      0.735     1.000
control: carry <- carry (expect UNCHANGED)               200     0.940      0.915     0.995
done in 82s


,subspace_dim,runs,zero_node_runs,train_mean,train_sd,heldout_mean,heldout_sd,reverse_mean,reverse_sd,no_carry_control_mean,carry_control_mean
0,1,5,5,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
1,4,5,2,0.436667,0.399416,0.531111,0.048572,0.3600,0.128755,0.79500,0.898333
2,8,5,1,0.636667,0.360123,0.593333,0.035590,0.5225,0.051307,0.83625,0.875000
3,16,5,0,0.914000,0.056391,0.669333,0.112309,0.3740,0.149562,0.77700,0.902000


,node,selected_count,selection_frequency
1,"(6, 7)",3,0.15
5,"(12, 7)",3,0.15
4,"(11, 7)",2,0.10
7,"(14, 7)",2,0.10
3,"(10, 7)",1,0.05
2,"(8, 7)",1,0.05
0,"(4, 7)",1,0.05
6,"(13, 7)",1,0.05


,jaccard
count,162.000000
mean,0.040123
std,0.184647
min,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,1.000000


In [ ]:
# Negative-control localization experiment.
# It optimizes a mask/subspace using only candidate positions before the ones-digit token.
# This is a separate search and can take several minutes; leave disabled until needed.
RUN_NEGATIVE_CONTROL_SEARCH = True
if RUN_NEGATIVE_CONTROL_SEARCH:
    early_nodes = [(layer, pos) for layer, pos in nodes if pos in (5, 6)]
    early_out = run_carry_search(
        model, early_nodes, n_samples=300, seed=777, amp=True,
        subspace_dim=8, lambda_=0.0095, K=4.0, n_steps=300,
        batch_size=64, micro_batch=32, lr_alpha=3e-2, lr_basis=1e-2)
    if not early_out["result"]["selected_nodes"]:
        print("No early nodes selected. This is an empty negative-control discovery under the current hyperparameters; validation is not run because there is no candidate to patch.")
    else:
        early_report = validate_search(model, early_out, early_nodes,
                                       n_heldout=300, n_controls=200,
                                       refine_steps=100, seed=1777, amp=True)
        display(early_report["table"])
else:
    print("Negative-control search skipped by default. Set RUN_NEGATIVE_CONTROL_SEARCH=True to run; an empty selection is reported without raising an exception.")


candidates=384 | base acc=0.964 CE=0.228 | source acc=0.917 CE=0.400 | both-correct=347/384
data+cache ready in 4s
no-intervention acc: 0.0


  0%|          | 0/300 [00:00<?, ?it/s]

~0.5s/step -> ~2.6 min total
selected 0/32 nodes
  (no nodes selected)
ZERO-NODE DIAGNOSTIC: no alpha crossed the hard threshold alpha > 0.
alpha range=[-1.9011, -1.8807], max inclusion probability=0.0005
Top candidates (layer, position, alpha, soft inclusion probability):
  (11, 6): alpha=-1.8807, p=0.0005
  (13, 5): alpha=-1.8880, p=0.0005
  (14, 6): alpha=-1.8883, p=0.0005
  (8, 6): alpha=-1.8887, p=0.0005
  (6, 5): alpha=-1.8889, p=0.0005
  (15, 6): alpha=-1.8891, p=0.0005
  (5, 5): alpha=-1.8898, p=0.0005
  (2, 6): alpha=-1.8902, p=0.0005
  (3, 6): alpha=-1.8902, p=0.0005
  (13, 6): alpha=-1.8903, p=0.0005
Treat this as a failed/empty discovery for these hyperparameters; do not invent a node or interpret it as proof that the feature is absent.
FOUND mask IIA (source tens acc, train pairs): 0.000
done in 158s
No early nodes selected. This is an empty negative-control discovery under the current hyperparameters; validation is not run because there is no candidate to patch.


In [ ]:
# LOCKED FINAL EVALUATION — manually set True only after the candidate is frozen.
# Choose `final_out` from the selected search result, then refine on training data only if desired.
# Do NOT choose subspace_dim/nodes/seed based on this final benchmark's results.
RUN_LOCKED_FINAL_EVAL = True
if RUN_LOCKED_FINAL_EVAL:
    # Explicitly choose the already-frozen candidate here. Do not choose based on the locked-test scores.
    if "out8" not in globals():
        raise RuntimeError("Run the selected development search first and explicitly point final_out to that result.")
    final_out = out8
    chosen = torch.nonzero(final_out["result"]["mask_hard"]).flatten().tolist()
    if not chosen:
        raise RuntimeError("The chosen candidate has zero selected nodes; locked evaluation is undefined. Select a non-empty candidate using development data only.")
    final_nodes = [nodes[i] for i in chosen]
    final_Q = final_out["result"]["Q"][chosen].to(device)
    digits = get_digit_ids(model)
    train_gold = [digits[r[3]] for r in final_out["data"]]
    train_prep = prepare(model, final_out["data"], final_nodes, train_gold, amp=True)
    final_Q = refine_Q(model, train_prep, final_Q, n_steps=100, lr=2e-3,
                       batch_size=64, micro_batch=32, amp=True, seed=2026)
    locked = build_locked_benchmark(model, final_out["data"], n_forward=300,
                                    n_controls=200, seed=42000, amp=True)
    final_report = score_locked_benchmark(model, locked, final_nodes, final_Q,
                                          amp=True, random_trials=10,
                                          include_full_patch=True, bootstrap_seed=2026)
    display(final_report["table"])
else:
    print("Locked final evaluation is skipped by default. Freeze a non-empty candidate, then set RUN_LOCKED_FINAL_EVAL=True.")


refine Q:   0%|          | 0/100 [00:00<?, ?it/s]

candidates=768 | base acc=0.966 CE=0.193 | source acc=0.913 CE=0.372 | both-correct=693/768
Locked benchmark pair counts: {'forward': 300, 'reverse': 300, 'control_no_carry': 200, 'control_carry': 200, 'digit_ids': 10}
Individual prompts reused across train/test: 0
Note: forward/reverse sets intentionally reuse the same pair in opposite directions.
forward: learned - no_intervention: +0.533 (paired bootstrap 95% CI +0.477, +0.590)
forward: learned - full_residual_patch: -0.007 (paired bootstrap 95% CI -0.057, +0.043)
forward: learned - without_node_(6, 7): +0.533 (paired bootstrap 95% CI +0.477, +0.590)
reverse: learned - no_intervention: +0.457 (paired bootstrap 95% CI +0.400, +0.513)
reverse: learned - full_residual_patch: -0.213 (paired bootstrap 95% CI -0.260, -0.167)
reverse: learned - without_node_(6, 7): +0.457 (paired bootstrap 95% CI +0.400, +0.513)
control_no_carry: learned - no_intervention: -0.215 (paired bootstrap 95% CI -0.275, -0.160)
control_no_carry: learned - full_res

,set,method,n,correct,accuracy,ci95_low,ci95_high,note
0,no-carry -> carry (expect source tens digit),learned_subspace,300,160.0,0.533333,4.768149e-01,0.589009,NaN
1,no-carry -> carry (expect source tens digit),no_intervention,300,0.0,0.000000,8.673617e-19,0.012643,NaN
2,no-carry -> carry (expect source tens digit),full_residual_patch,300,162.0,0.540000,4.834517e-01,0.595537,NaN
3,no-carry -> carry (expect source tens digit),"without_node_(6, 7)",300,0.0,0.000000,8.673617e-19,0.012643,NaN
4,no-carry -> carry (expect source tens digit),random_subspace (mean over trials),300,NaN,0.000000,0.000000e+00,0.000000,"interval is across random-direction trials, no..."
5,carry -> no-carry (expect original base tens d...,learned_subspace,300,137.0,0.456667,4.012029e-01,0.513226,NaN
6,carry -> no-carry (expect original base tens d...,no_intervention,300,0.0,0.000000,8.673617e-19,0.012643,NaN
7,carry -> no-carry (expect original base tens d...,full_residual_patch,300,201.0,0.670000,6.149359e-01,0.720766,NaN
8,carry -> no-carry (expect original base tens d...,"without_node_(6, 7)",300,0.0,0.000000,8.673617e-19,0.012643,NaN
9,carry -> no-carry (expect original base tens d...,random_subspace (mean over trials),300,NaN,0.001000,0.000000e+00,0.003333,"interval is across random-direction trials, no..."


## Reporting checklist

Before making a causal claim, save and report:

- Model and tokenizer version, library versions, GPU, exact code revision, all random seeds, candidate-node list, optimizer settings, and train/development/locked-test counts.
- Accuracy with Wilson 95% intervals for each test set and method, paired-bootstrap CIs for learned-vs-control differences, and per-seed results (not only the mean).
- Number of selected nodes, selected-node frequency, pairwise Jaccard overlap, and results of every negative control and ablation.
- The percentage of candidate examples rejected by model-correctness screening, and separate unfiltered model arithmetic accuracy where feasible.
- The locked test should be run only after all choices are fixed. A failed locked test is still the result; do not repeatedly revise the method against it.

A convincing result would require all these tests to agree. A higher training score or a larger basis alone is not evidence of a better causal abstraction.


## Interpreting zero-node runs (important)

The saved outputs in the earlier run show two different situations:

- The full-node search with `subspace_dim=1` selected `0/96` nodes. The sweep then raised `ValueError` because validation assumes a non-empty candidate. This is a failed/empty discovery for that setting, not evidence that no arithmetic information exists. The sweep cell is patched to record the empty result and continue to the remaining dimensions.
- The early-position negative control selected `0/32` nodes. That can be consistent with the timing hypothesis, but it is not conclusive: sparsity, initialization, optimizer dynamics, and the chosen dimension can also cause an empty mask. The patched cell reports this without trying to validate a nonexistent candidate.
- The locked-final-evaluation cell has no saved output in the uploaded notebook; it was not executed. It previously defaulted to `True` and reused `out8`, which could silently run a stale candidate. It is now opt-in and refuses to evaluate an empty candidate.

When a search selects no nodes, inspect the printed `alpha` and soft inclusion probabilities. If all logits stay below zero, the hard threshold `alpha > 0` returns an empty mask. Do not simply force the top-alpha node into the solution; instead tune sparsity and optimization on development data, then rerun across multiple seeds and report the empty runs too.
